#Part 1: The Movie Database API

This section of the Colab Notebook is focused on collecting the data from The Movie Database API. Running it can take several minutes as the data has to be processed. For future scenarios, after running this notebook once, a csv file will be downlaoded (it is also available in the github repository so you can skip the API section all together). If you simply want to run the notebook without the API, start at Part 2.

In [19]:
import os
import requests
import random
import time
import json
from google.colab import userdata

os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")
TMDB_API_KEY = os.environ["TMDB_API_KEY"]  # TMBD API Key stored in environment variable

BASE_URL = "https://api.themoviedb.org/3"

session = requests.Session()
session.params = {"api_key": TMDB_API_KEY}
random.seed(10)

In [20]:
def api_request(endpoint, params=None):
    for attempt in range(5):
      response = session.get(
          f"{BASE_URL}/{endpoint}",
          params=params,
          timeout=30
      )

      if response.status_code >= 500:
        time.sleep(2 ** attempt)
        continue

      response.raise_for_status()
      return response.json()

    raise Exception(f"Request failed: {endpoint}")

In [21]:
def discover_movies(start_year, end_year, pages_per_year):
  movie_ids = set()
  for year in range(start_year, end_year + 1):
    params = {
        "primary_release_year": year,
        "with_original_language": "en",
        "sort_by": "popularity.desc",
        "vote_count.gte": 20,
        "include_adult": False,
        "include_video": False
    }

    initial = api_request("discover/movie", params)
    total_pages = min(initial["total_pages"], 500)
    selected_pages = random.sample(
        range(1, total_pages + 1),
        min(pages_per_year, total_pages)
    )

    for page in selected_pages:
      params["page"] = page
      data = api_request("discover/movie", params)
      for movie in data["results"]:
        movie_ids.add(movie["id"])
    print(f"Year {year} completed. Total: {len(movie_ids)}")
  return sorted(movie_ids)


In [22]:
movie_ids = discover_movies(1970, 2026, 5)
print(f"Collected {len(movie_ids)} unique movies")

Year 1970 completed. Total: 91
Year 1971 completed. Total: 191
Year 1972 completed. Total: 273
Year 1973 completed. Total: 373
Year 1974 completed. Total: 471
Year 1975 completed. Total: 553
Year 1976 completed. Total: 653
Year 1977 completed. Total: 740
Year 1978 completed. Total: 840
Year 1979 completed. Total: 933
Year 1980 completed. Total: 1025
Year 1981 completed. Total: 1120
Year 1982 completed. Total: 1220
Year 1983 completed. Total: 1320
Year 1984 completed. Total: 1420
Year 1985 completed. Total: 1517
Year 1986 completed. Total: 1617
Year 1987 completed. Total: 1717
Year 1988 completed. Total: 1817
Year 1989 completed. Total: 1906
Year 1990 completed. Total: 2006
Year 1991 completed. Total: 2106
Year 1992 completed. Total: 2206
Year 1993 completed. Total: 2306
Year 1994 completed. Total: 2405
Year 1995 completed. Total: 2491
Year 1996 completed. Total: 2591
Year 1997 completed. Total: 2691
Year 1998 completed. Total: 2791
Year 1999 completed. Total: 2891
Year 2000 completed. 

In [23]:
# Save IDs
with open("movie_ids.json", "w") as f:
  json.dump(movie_ids, f)

In [24]:
data = api_request(
    "movie/550",
    {"append_to_response": "credits"}
)

params = {
    "append_to_resoonse": "credits,keywords,release_dates"
}

In [25]:
def get_movie_details(movie_id):
  data = api_request(
      f"movie/{movie_id}",
      {"append_to_response": "credits,keywords,release_dates"}
  )

  if data is None:
    return None

  cast = data["credits"]["cast"]
  crew = data["credits"]["crew"]
  directors = [
      person["name"]
      for person in crew
      if person["job"] == "Director"
  ]

  actors = [
      person["name"]
      for person in cast[:5] # top 5
  ]

  return {
      "id": data["id"],
      "title": data["title"],
      "budget": data["budget"],
      "revenue": data["revenue"],
      "runtime": data["runtime"],
      "release_date": data["release_date"],
      "original_language": data["original_language"],
      "popularity": data["popularity"],
      "vote_average": data["vote_average"],
      "vote_count": data["vote_count"],
      "overview": data["overview"],
      "genres": [
          g["name"] for g in data["genres"]
      ],
      "production_companies": [
          c["name"] for c in data["production_companies"]
      ],
      "production_countries": [
          c["name"] for c in data["production_countries"]
      ],
      "directors": directors,
      "actors": actors,
      "cast_size": len(cast),
      "crew_size": len(crew),
      "keywords": [
          k["name"]
          for k in data["keywords"]["keywords"]
      ]
  }

In [26]:
with open("movies_raw_jsonl", "a+") as file:
  file.seek(0)
  completed = {
      json.loads(line)["id"]
      for line in file
      if line.strip()
  }

  for i, movie_id in enumerate(movie_ids):
    if movie_id in completed:
      continue

    movie = get_movie_details(movie_id)

    if movie is not None:
      file.write(json.dumps(movie) + "\n")
      file.flush()

    if (i + 1) % 100 == 0:
      print(f"Processed {i + 1}/{len(movie_ids)}")
    time.sleep(0.1)


Processed 200/5569
Processed 300/5569
Processed 400/5569
Processed 500/5569
Processed 600/5569
Processed 700/5569
Processed 800/5569
Processed 900/5569
Processed 1000/5569
Processed 1100/5569
Processed 1200/5569
Processed 1300/5569
Processed 1400/5569
Processed 1500/5569
Processed 1600/5569
Processed 1700/5569
Processed 1800/5569
Processed 1900/5569
Processed 2000/5569
Processed 2100/5569
Processed 2200/5569
Processed 2300/5569
Processed 2400/5569
Processed 2500/5569
Processed 2600/5569
Processed 2700/5569
Processed 2800/5569
Processed 2900/5569
Processed 3000/5569
Processed 3100/5569
Processed 3200/5569
Processed 3300/5569
Processed 3400/5569
Processed 3500/5569
Processed 3600/5569
Processed 3700/5569
Processed 3800/5569
Processed 3900/5569
Processed 4000/5569
Processed 4100/5569
Processed 4200/5569
Processed 4300/5569
Processed 4400/5569
Processed 4500/5569
Processed 4600/5569
Processed 4700/5569
Processed 4800/5569
Processed 4900/5569
Processed 5000/5569
Processed 5100/5569
Processe

In [29]:
import pandas as pd
from google.colab import files

df = pd.read_json("movies_raw_jsonl", lines=True)
df = df.drop_duplicates(subset="id")
df.to_csv("movies_raw.csv", index=False)
files.download("movies_raw.csv")
print(df.shape)
df.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(5943, 19)


,id,title,budget,revenue,runtime,release_date,original_language,popularity,vote_average,vote_count,overview,genres,production_companies,production_countries,directors,actors,cast_size,crew_size,keywords
0,5,Four Rooms,4000000,4257354,98,1995-12-09,en,5.1600,5.900,2942,It's Ted the Bellhop's first night on the job....,[Comedy],"[Miramax, A Band Apart]",[United States],"[Allison Anders, Alexandre Rockwell, Robert Ro...","[Tim Roth, Jennifer Beals, David Proval, Ione ...",28,116,"[hotel, new year's eve, witch, bet, sperm, hot..."
1,6,Judgment Night,21000000,12136938,110,1993-10-15,en,2.2585,6.455,397,"Four young friends, while taking a shortcut en...","[Action, Crime, Drama, Thriller]","[Largo Entertainment, JVC]","[United States, Japan]",[Stephen Hopkins],"[Emilio Estevez, Cuba Gooding Jr., Denis Leary...",31,19,"[drug dealer, chicago, illinois, escape, one n..."
2,8,Life in Loops (A Megacities RMX),42000,0,80,2006-01-01,en,0.5527,7.300,34,Timo Novotny labels his new project an experim...,[Documentary],[inLoops],[Austria],[Timo Novotny],[],0,10,[megacities]
3,11,Star Wars,11000000,775398007,121,1977-05-25,en,30.8131,8.206,23222,Princess Leia is captured and held hostage by ...,"[Adventure, Action, Science Fiction]",[Lucasfilm Ltd.],[United States],[George Lucas],"[Mark Hamill, Harrison Ford, Carrie Fisher, Pe...",105,95,"[empire, galaxy, rebellion, android, hermit, s..."
4,12,Finding Nemo,94000000,940335536,100,2003-05-30,en,23.7034,7.823,21265,"Nemo, an adventurous young clownfish, is unexp...","[Animation, Family, Adventure]",[Pixar],[United States],[Andrew Stanton],"[Albert Brooks, Ellen DeGeneres, Alexander Gou...",69,423,"[fish, sydney, australia, parent child relatio..."
5,13,Forrest Gump,55000000,677387716,142,1994-06-23,en,32.6376,8.462,30966,A man with a low IQ has accomplished great thi...,"[Comedy, Drama, Romance]","[Paramount Pictures, The Steve Tisch Company, ...",[United States],[Robert Zemeckis],"[Tom Hanks, Robin Wright, Gary Sinise, Mykelti...",144,178,"[new year's eve, vietnam war, vietnam veteran,..."
6,14,American Beauty,15000000,356296601,122,1999-09-15,en,11.0447,7.996,13585,"Lester Burnham, a depressed suburban father in...",[Drama],"[DreamWorks Pictures, Jinks/Cohen Company]",[United States],[Sam Mendes],"[Kevin Spacey, Annette Bening, Thora Birch, We...",41,148,"[estate agent, adultery, depression, first tim..."
7,16,Dancer in the Dark,12500000,40061153,140,2000-09-01,en,5.1154,7.827,2121,"Selma, a Czech immigrant on the verge of blind...","[Drama, Crime]","[Zentropa Entertainments, DR, SVT Drama, ARTE,...","[Denmark, Finland, France, Germany, Iceland, N...",[Lars von Trier],"[Björk, Catherine Deneuve, David Morse, Peter ...",69,137,"[factory worker, individual, immigrant, blindn..."
8,20,My Life Without Me,2500000,12300000,106,2003-03-07,en,1.9630,6.100,521,A fatally ill mother with only two months to l...,"[Drama, Romance]","[El Deseo, Milestone Productions, Miss Wasabi]","[Canada, Spain]",[Isabel Coixet],"[Sarah Polley, Amanda Plummer, Scott Speedman,...",13,35,"[dying and death, daughter, farewell, night sh..."
9,22,Pirates of the Caribbean: The Curse of the Bla...,140000000,655011224,143,2003-07-09,en,23.6253,7.839,23145,When wily pirate Captain Barbossa seizes Jack ...,"[Adventure, Fantasy, Action]","[Walt Disney Pictures, Jerry Bruckheimer Films]",[United States],[Gore Verbinski],"[Johnny Depp, Geoffrey Rush, Orlando Bloom, Ke...",70,331,"[ship, blacksmith, gold, exotic island, govern..."


In [30]:
# Determine Missing Data
print("Total movies:", len(df))

print("Missing revenue:", (df["revenue"] == 0).sum())
print("Missing budget:", (df["budget"] == 0).sum())

print("Missing revenue percentage:",
      (df["revenue"] == 0).mean() * 100)

print("Missing budget percentage:",
      (df["budget"] == 0).mean() * 100)

Total movies: 5943
Missing revenue: 3676
Missing budget: 3615
Missing revenue percentage: 61.85428234898199
Missing budget percentage: 60.827864714790515
